# GeoGram: Klasifikace sg/pl — IJP ÚJČ (všechny -ice obce)

Notebook spustí parser ÚJČ na všech 1 806 obcích s koncovkou `-ice` a uloží výsledky.

**Výstup:** `data/processed/ice_grammar_ujc.csv`

**Důležité:** Běh trvá ~30–45 minut (throttling 1 req/s).
Notebook ukládá průběžné výsledky — pokud se přeruší, spusť znovu a pokračuje od checkpointu.

### Kategorie výsledků
| hodnota | význam |
|---|---|
| `singular` | pouze singulár (pomnožné sg) |
| `plural` | pouze plurál (pomnožné pl) |
| `both` | má obě formy |
| `not_found` | IJP ÚJČ nemá heslo nebo server nevrátil tabulku |
| `unknown` | tabulka nalezena, ale bez rozlišitelných hodnot |
| `error` | síťová/HTTP chyba |

## 1. Setup

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from geogram.ujc import fetch_ujc_number, SINGULAR, PLURAL, BOTH, UNKNOWN, NOT_FOUND

PROCESSED_DIR = project_root / 'data' / 'processed'
INPUT_CSV     = PROCESSED_DIR / 'municipalities_ice_integrated.csv'
CHECKPOINT    = PROCESSED_DIR / 'ujc_checkpoint.csv'
OUTPUT_CSV    = PROCESSED_DIR / 'ice_grammar_ujc.csv'

SLEEP_BETWEEN_REQUESTS = 1.5   # sekundy mezi požadavky
CHECKPOINT_EVERY       = 50    # uložit checkpoint každých N obcí

# Pokud OUTPUT_CSV existuje z předchozího běhu, znovu zpracovat tyto kategorie.
# Nastav na prázdnou množinu pokud chceš začít úplně od začátku.
REPROCESS_CATEGORIES = {'unknown', 'error', 'not_found'}

# Varování: pokud server vrátí NOT_FOUND X-krát za sebou, pravděpodobně rate-limituje.
CONSECUTIVE_NOT_FOUND_WARN = 10

print(f'Input:      {INPUT_CSV}')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Output:     {OUTPUT_CSV}')
print(f'Re-procesovat: {REPROCESS_CATEGORIES}')

## 2. Načtení dat a resumování checkpointu

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Načteno {len(df):,} obcí z {INPUT_CSV.name}')

# Priorita: checkpoint > výstupní CSV > začít od začátku
if CHECKPOINT.exists():
    done = pd.read_csv(CHECKPOINT)
    done_codes = set(done[~done['ujc_number'].isin(REPROCESS_CATEGORIES)]['code'])
    print(f'Checkpoint nalezen: {len(done):,} záznamů, '
          f'{len(done_codes):,} definitivních (zbytek bude re-procesován).')
elif OUTPUT_CSV.exists() and REPROCESS_CATEGORIES:
    prev = pd.read_csv(OUTPUT_CSV)[['code', 'name', 'ujc_number']]
    done = prev[~prev['ujc_number'].isin(REPROCESS_CATEGORIES)]
    done_codes = set(done['code'])
    reprocess_n = len(prev) - len(done)
    print(f'Předchozí výstup nalezen: {len(prev):,} záznamů.')
    print(f'  Definitivní (přeskočím): {len(done):,}')
    print(f'  Re-procesovat ({REPROCESS_CATEGORIES}): {reprocess_n:,}')
else:
    done = pd.DataFrame(columns=['code', 'name', 'ujc_number'])
    done_codes = set()
    print('Začínám od začátku.')

todo = df[~df['code'].isin(done_codes)].reset_index(drop=True)
print(f'\nZbývá zpracovat: {len(todo):,} obcí')

## 3. Klasifikace (s throttlingem a checkpointováním)

In [ ]:
results = done.to_dict('records')
errors = []
consecutive_not_found = 0

pbar = tqdm(todo.iterrows(), total=len(todo), unit='obec',
            desc='ÚJČ klasifikace', dynamic_ncols=True)

for i, row in pbar:
    name = row['name']
    code = row['code']

    try:
        number = fetch_ujc_number(name)
    except Exception as exc:
        number = 'error'
        errors.append({'code': code, 'name': name, 'error': str(exc)})

    results.append({'code': code, 'name': name, 'ujc_number': number})

    # Detekce rate-limitingu: příliš mnoho NOT_FOUND za sebou = server nereaguje správně
    if number == NOT_FOUND:
        consecutive_not_found += 1
        if consecutive_not_found == CONSECUTIVE_NOT_FOUND_WARN:
            pbar.write(f'\n[VAROVÁNÍ] {consecutive_not_found} NOT_FOUND v řadě — '
                       f'server pravděpodobně rate-limituje. Zpomaluju na 3s.')
    else:
        consecutive_not_found = 0

    sleep_time = 3.0 if consecutive_not_found >= CONSECUTIVE_NOT_FOUND_WARN else SLEEP_BETWEEN_REQUESTS
    pbar.set_postfix({'poslední': name[:20], 'výsledek': number,
                      'chyby': len(errors), 'nf_v_řadě': consecutive_not_found})

    # Checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False, encoding='utf-8')

    time.sleep(sleep_time)

pbar.close()
print(f'\nHotovo. Chyby: {len(errors)}')
if errors:
    for e in errors:
        print(f"  {e['name']}: {e['error']}")

## 4. Uložení výsledků

In [ ]:
df_results = pd.DataFrame(results)

# Spojit s původními daty (demografie, region atd.)
df_final = df.merge(df_results[['code', 'ujc_number']], on='code', how='left')

df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
# Smazat checkpoint — výsledky jsou uloženy v output
if CHECKPOINT.exists():
    CHECKPOINT.unlink()

print(f'Uloženo {len(df_final):,} řádků do {OUTPUT_CSV}')
df_final[['name', 'region_name', 'population_total', 'ujc_number']].head(10)

## 5. Základní statistiky

In [ ]:
counts = df_final['ujc_number'].value_counts()
total = len(df_final)

print('=== Distribuce gramatického čísla (ÚJČ) ===')
for label, n in counts.items():
    print(f'  {label:<12}: {n:>5}  ({n/total*100:.1f} %)')

classified = df_final['ujc_number'].isin([SINGULAR, PLURAL, BOTH]).sum()
print(f'\nCelkem obcí:     {total:,}')
print(f'Klasifikováno:   {classified:,} ({classified/total*100:.1f} %)')
print(f'Bez hesla (IJP): {counts.get(NOT_FOUND, 0):,}')
print(f'Chyby sítě:      {counts.get("error", 0):,}')

In [ ]:
# Rozložení sg/pl po krajích
df_known = df_final[df_final['ujc_number'].isin([SINGULAR, PLURAL, BOTH])]

pivot = (
    df_known
    .groupby(['region_name', 'ujc_number'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .sort_values('total', ascending=False)
)

print('=== Sg/pl po krajích ===')
print(pivot.to_string())

In [ ]:
# Příklady z každé kategorie (top 5 podle populace)
for label in [SINGULAR, PLURAL, BOTH, NOT_FOUND, UNKNOWN, 'error']:
    subset = df_final[df_final['ujc_number'] == label]
    if subset.empty:
        continue
    top = subset.nlargest(5, 'population_total')[['name', 'region_name', 'population_total']]
    print(f'\n--- {label.upper()} (top 5 podle populace) ---')
    print(top.to_string(index=False))